In [ ]:
# =========================================================
# 01_organizar_dados_inmet.ipynb - Edivan Silva
# =========================================================
# Objetivo:
# - Extrair, ler e organizar dados do INMET
# - Gerar planilhas resumidas e pivotadas
# - Calcular porcentagem de dados disponíveis
# - Filtrar estações pelo período de estudo (1980-2023)
# =========================================================

# =========================================================
# 0) CONFIGURAÇÕES
# =========================================================
ANO_INICIAL = 1980
ANO_FINAL   = 2023
PASTA_RAR   = "/content/estacoesNEB.rar"
PASTA_DADOS = "/content/"

!pip install rarfile --quiet

# =========================================================
# 1) IMPORTS
# =========================================================
import rarfile, os, pandas as pd
from google.colab import files

# =========================================================
# 2) EXTRAIR ARQUIVOS
# =========================================================
with rarfile.RarFile(PASTA_RAR) as rf:
    rf.extractall(PASTA_DADOS)

In [ ]:
# =========================================================
# 3) LEITURA DOS CSVs
# =========================================================
resumo_estacoes = []
series = []

for arquivo in os.listdir("/content/estacoesNEB"):
    if not arquivo.endswith(".csv"):
        continue
    caminho = os.path.join("/content/estacoesNEB", arquivo)
    with open(caminho, encoding="latin1") as f:
        linhas = f.readlines()

    # METADADOS
    nome = linhas[0].replace("Nome:", "").strip()
    codigo = linhas[1].split(":")[1].strip()
    latitude = float(linhas[2].split(":")[1].strip())
    longitude = float(linhas[3].split(":")[1].strip())
    altitude = float(linhas[4].split(":")[1].strip())
    situacao = linhas[5].split(":")[1].strip()
    ano_ini = int(linhas[6].split(":")[1][:4])
    ano_fim = int(linhas[7].split(":")[1][:4])

    resumo_estacoes.append([codigo, nome, latitude, longitude, altitude, situacao, ano_ini, ano_fim])

    # LINHA DE INÍCIO DOS DADOS
    linha_inicio = next((i for i,l in enumerate(linhas) if l.startswith("Data Medicao")), None)
    if linha_inicio is None:
        continue

    df = pd.read_csv(caminho, sep=";", skiprows=linha_inicio, usecols=[0,1], names=["data","precipitacao"], keep_default_na=False)
    df["data"] = pd.to_datetime(df["data"], errors="coerce")
    df = df.dropna(subset=["data"])
    df["Ano"] = df["data"].dt.year
    df["Mes"] = df["data"].dt.month
    df["Nome"] = nome
    df["Lat"] = latitude
    df["Long"] = longitude
    series.append(df[["Nome","Lat","Long","Ano","Mes","precipitacao"]])

# =========================================================
# 4) CRIAR PLANILHA RESUMO DAS ESTAÇÕES COM %
# =========================================================
df_resumo = pd.DataFrame(resumo_estacoes, columns=["Codigo","Nome","Latitude","Longitude","Altitude_m","Situacao","Ano_Inicial","Ano_Final"])
df_mensal = pd.concat(series, ignore_index=True)

# % de dados disponíveis
def perc_valido(df_est):
    total = len(df_est)
    if total == 0: return 0
    return df_est["precipitacao"].apply(lambda x: str(x).lower() not in ["null",""]).sum() / total * 100

df_resumo["Porcentagem_Concluida"] = df_resumo["Nome"].apply(lambda n: perc_valido(df_mensal[df_mensal["Nome"]==n]))
df_resumo.to_excel("01_RESUMO_ESTACOES.xlsx", index=False)
files.download("01_RESUMO_ESTACOES.xlsx")

# =========================================================
# 5) BASE MENSAL LONGA
# =========================================================
df_mensal.rename(columns={"precipitacao":"Precipitacao"}, inplace=True)
df_mensal.to_excel("02_BASE_MENSAL_LONGA.xlsx", index=False)
files.download("02_BASE_MENSAL_LONGA.xlsx")

# =========================================================
# 6) BASE PIVOTADA (JAN-DEZ + TOTAL)
# =========================================================
df_pivot = df_mensal.pivot_table(index=["Nome","Lat","Long","Ano"], columns="Mes", values="Precipitacao", aggfunc=lambda x:x).reset_index()
meses = {1:"JAN",2:"FEV",3:"MAR",4:"ABR",5:"MAI",6:"JUN",7:"JUL",8:"AGO",9:"SET",10:"OUT",11:"NOV",12:"DEZ"}
df_pivot.rename(columns=meses, inplace=True)

# Coluna TOTAL
df_pivot["TOTAL"] = df_pivot[list(meses.values())].apply(lambda row: sum([float(v) for v in row if str(v).strip() not in ["","null"]]), axis=1)
df_pivot.to_excel("03_INMET_MENSAL_TABELADO.xlsx", index=False)
files.download("03_INMET_MENSAL_TABELADO.xlsx")

# =========================================================
# 7) FILTRAR PELO PERÍODO DE ESTUDO
# =========================================================
estacoes_validas = df_resumo[(df_resumo["Ano_Inicial"]<=ANO_INICIAL) & (df_resumo["Ano_Final"]>=ANO_FINAL)]

df_mensal_estudo = df_mensal[(df_mensal["Nome"].isin(estacoes_validas["Nome"])) & (df_mensal["Ano"].between(ANO_INICIAL, ANO_FINAL))]
df_pivot_estudo = df_pivot[(df_pivot["Nome"].isin(estacoes_validas["Nome"])) & (df_pivot["Ano"].between(ANO_INICIAL, ANO_FINAL))]

df_mensal_estudo.to_excel("04_BASE_MENSAL_LONGA_ESTUDO.xlsx", index=False)
df_pivot_estudo.to_excel("05_INMET_MENSAL_TABELADO_ESTUDO.xlsx", index=False)
files.download("04_BASE_MENSAL_LONGA_ESTUDO.xlsx")
files.download("05_INMET_MENSAL_TABELADO_ESTUDO.xlsx")

# Resumo das estações do período de estudo com %
estacoes_validas["Porcentagem_Concluida"] = estacoes_validas["Nome"].apply(lambda n: perc_valido(df_mensal_estudo[df_mensal_estudo["Nome"]==n]))
estacoes_validas.to_excel("06_RESUMO_ESTACOES_ESTUDO.xlsx", index=False)
files.download("06_RESUMO_ESTACOES_ESTUDO.xlsx")
